In [4]:
print(123)

123


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)


In [7]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(
    query_vector, 
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
    )

In [8]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

In [12]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
anthropic_client = Anthropic()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [14]:

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=anthropic_client,
)

In [15]:
vector_assistant.rag("the program has already begun, can I still sign up?")

("Based on the course information provided, **yes, you can still sign up even though the program has already begun!**\n\nHere are the key points:\n\n1. **You can join anytime** - The course accepts late joiners\n2. **Certificate eligibility** - If you want to receive a certificate, you'll need to submit your project while submissions are still being accepted\n3. **No confirmation email needed** - You don't need to wait for a confirmation email to start. You're accepted and can begin learning and submitting homework right away (while the submission form is open)\n4. **Registration is informal** - Registration is just used to gauge interest and isn't checked against a registered list, so you can start without formally registering if you prefer\n\nSo go ahead and sign up! Just keep in mind the deadline for project submissions if you want to earn a certificate.",
 Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, ca

In [16]:
vs_index.close()